# SFT HunyuanOCR-1.5 base — Colab

Chạy từ trên xuống. Runtime: **A100** (`Runtime → Change runtime type`).
Chi tiết và cách xử lý sự cố: [`docs/COLAB.md`](https://github.com/jajqja/custom-hunyuan-ocr/blob/main/docs/COLAB.md).

Trước khi chạy, sửa hai biến ở cell **Cấu hình**.

## 1. GPU và PACK_LEN

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
PACK_LEN = 16384 if GB >= 70 else 8192 if GB >= 35 else 4096
print(f"{GB:.0f} GB -> PACK_LEN={PACK_LEN}")
# Colab cấp A100 cả bản 40GB lẫn 80GB tuỳ lần, nên đừng đặt cứng: 80GB -> 16384,
# 40GB -> 8192 (~2 trang/pack, bình thường). Trên 80GB có thể thử 20480 sau khi
# xem headroom ở lần chạy đầu.

## 2. Cấu hình — **sửa hai dòng dưới**

In [ ]:
DATASET_REPO = "<user>/vietnamese-doc-ocr"   # repo dataset đã đẩy lên HF
OUTPUT_REPO  = "<user>/hunyuanocr-vi-sft"    # nơi đẩy model sau khi train (bước 9)

REPO_URL = "https://github.com/jajqja/custom-hunyuan-ocr.git"
MODEL_DIR, DATA_DIR, WORK = "/content/HunyuanOCR", "/content/dataset", "/content/hyocr"
DRIVE_DIR = "/content/drive/MyDrive/hyocr_sft"
RUN_NAME = "colab_run"

## 3. Gắn Drive

Session Colab chết là mất sạch `/content`. **Không** ghi checkpoint thẳng vào
Drive — một checkpoint model 1B kèm optimizer state cỡ 8–12 GB, ghi qua FUSE
sẽ làm training đứng hình. Bước 7 lưu vào `/content` rồi rsync sang Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(DRIVE_DIR, exist_ok=True)

## 4. Clone fork

In [ ]:
!git clone -q $REPO_URL $WORK
%cd $WORK
!git log --oneline -1

## 5. Dependency

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate deepspeed safetensors \
                   binpacking lmdb tensorboard "huggingface_hub>=0.34"

`flash-attn` bắt buộc: `train/trainer.py:3` import `flash_attn_varlen_func`
ngay đầu module và `train_hunyuan.py` truyền cứng `attn_implementation="flash_attention_2"`.
Build from source mất 40–60 phút, nên lấy wheel dựng sẵn khớp runtime.

In [ ]:
import subprocess, sys, torch, urllib.request

tv  = ".".join(torch.__version__.split("+")[0].split(".")[:2])
cu  = "cu" + torch.version.cuda.split(".")[0]
abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
py  = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"runtime: torch {tv} / {cu} / cxx11abi{abi} / {py}")

BASE = "https://github.com/Dao-AILab/flash-attention/releases/download"
def wheel_url(v):
    return (f"{BASE}/v{v}/flash_attn-{v}+{cu}torch{tv}"
            f"cxx11abi{abi}-{py}-{py}-linux_x86_64.whl")

whl = None
for v in ("2.8.3", "2.8.2", "2.8.1", "2.8.0", "2.7.4.post1"):
    u = wheel_url(v)
    try:
        urllib.request.urlopen(urllib.request.Request(u, method="HEAD"), timeout=20)
        whl = u
        break
    except Exception:
        print("  không có:", u.rsplit("/", 1)[1])

assert whl, ("Không có wheel khớp runtime này. Chọn tay ở "
             "https://github.com/Dao-AILab/flash-attention/releases theo 4 mảnh in ra trên.")
print("cài:", whl)
subprocess.run(["pip", "install", "-q", whl], check=True)

In [ ]:
from flash_attn.flash_attn_interface import flash_attn_varlen_func
print("flash-attn OK")

## 6. Đăng nhập HF, tải model + dataset

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # token quyền read; dataset đang private

In [ ]:
!hf download tencent/HunyuanOCR --local-dir $MODEL_DIR --exclude "v1.0/*"
!hf download $DATASET_REPO --repo-type dataset --local-dir $DATA_DIR
!ls $DATA_DIR

Prompt không nằm thành file riêng trong repo dataset — nó ở trong
`README.md`, trong khối ```` ```text ````.

In [ ]:
import re, pathlib

readme = pathlib.Path(f"{DATA_DIR}/README.md").read_text(encoding="utf-8")
prompt = re.search(r"````text\n(.*?)\n````", readme, re.S).group(1).strip()
pathlib.Path("/content/ocr_prompt.md").write_text(prompt, encoding="utf-8")
print(prompt)

## 7. Convert sang raw JSONL

In [ ]:
!python tools/makedata_to_hyocr.py \
    --root $DATA_DIR \
    --prompt-file /content/ocr_prompt.md \
    --out-dir $WORK/data/raw \
    --data-list $WORK/data/data_list.txt
# cột "thiếu ảnh" phải là 0

## 8. Pack

`NUM_PROCESSES=2` chứ không phải 32: Colab chỉ có ~12 vCPU và mỗi process load
một processor riêng. Mất 10–20 phút vì phải mở từng ảnh để đếm vision token.

In [ ]:
!MODEL_PATH=$MODEL_DIR \
 INPUT_LIST=$WORK/data/data_list.txt \
 PACK_OUTPUT=$WORK/data/packed/train_$PACK_LEN.jsonl \
 PACK_LEN=$PACK_LEN \
 NUM_PROCESSES=2 THREADS_PER_PROCESS=4 \
 FOREGROUND=1 \
     bash scripts/pack_data.sh

In [ ]:
import json

n = t = 0
for line in open(f"{WORK}/data/packed/train_{PACK_LEN}.jsonl"):
    p = json.loads(line); n += 1; t += sum(x["num_tokens"] for x in p)
print(f"{n} pack, trung bình {t/n:.0f} token/pack")

!mkdir -p $DRIVE_DIR/packed && cp $WORK/data/packed/*.jsonl $DRIVE_DIR/packed/

## 9. Train

In [ ]:
import subprocess
# Đồng bộ output sang Drive mỗi 10 phút để session chết không mất checkpoint.
subprocess.Popen(f"while true; do rsync -a --delete {WORK}/output/ {DRIVE_DIR}/; "
                 f"sleep 600; done", shell=True)
print("rsync nền đã chạy")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $WORK/output

In [ ]:
!MODEL_PATH=$MODEL_DIR \
 TRAIN_DATA=$WORK/data/packed/train_$PACK_LEN.jsonl \
 PACK_LEN=$PACK_LEN \
 RUN_NAME=$RUN_NAME \
 SAVE_STEPS=25 \
     bash scripts/sft_base_1gpu.sh

OOM trên 40GB thì thử theo thứ tự: `TUNE_VISION=False` (đóng băng vision
tower, rẻ nhất) → `PACK_LEN=4096` (phải pack lại) → `DEEPSPEED=scripts/zero2.json`.

## 10. Mất session → chạy lại

Làm lại cell 1–7, rồi kéo checkpoint từ Drive về **trước** khi chạy lại cell
train với đúng `RUN_NAME` cũ. `train_hunyuan.py:221` tự
`resume_from_checkpoint=True` khi thấy `checkpoint-*` trong output dir.

In [ ]:
!mkdir -p $WORK/output $WORK/data/packed
!rsync -a $DRIVE_DIR/ $WORK/output/
!cp $DRIVE_DIR/packed/*.jsonl $WORK/data/packed/ 2>/dev/null
!ls $WORK/output/$RUN_NAME

## 11. Đẩy model đã train lên HF

In [ ]:
!hf upload $OUTPUT_REPO $WORK/output/$RUN_NAME . --repo-type model --private